# CSTL v3 — Plan d'Action Reproductible

Ce notebook exécute les 18 étapes du plan d'action CSTL v3 dans l'ordre, avec des points de décision explicites.

**Auteur** : Olivier Goyette
**Date** : avril 2026
**Version harness** : 1.1.0

---

## Ordre d'exécution

| Phase | Cellules | Durée | Coût |
|---|---|---|---|
| **0. Setup** | 1-3 | 2 min | $0 |
| **1a. Harness sanity (mock)** | 4-5 | 1 min | $0 |
| **1b. Pilote Claude (10 payloads)** | 6-8 | 3 min | ~$0.10 |
| **1c. Point de décision GO/NO-GO** | 9 | — | — |
| **1d. Production run (3 modèles)** | 10-11 | 30-60 min | ~$15-30 |
| **2. Analyse & figures** | 12-16 | 5 min | $0 |
| **3. Export livrables** | 17-18 | 1 min | $0 |

---

## Phase 0 — Setup

Installation des dépendances et récupération du script `run_experiments.py`.

In [ ]:
# Cellule 1 — Dépendances
# Le harness lui-même n'a besoin que de la stdlib Python.
# Les SDK LLM ne sont nécessaires que pour les runs contre API réelle.

!pip install --quiet anthropic openai google-generativeai pandas matplotlib
print('✓ Dépendances installées')

In [ ]:
# Cellule 2 — Vérification du script
# Le fichier run_experiments.py doit être présent dans le répertoire courant.
# Si tu utilises Google Drive, monte-le ici. Sinon, upload manuel via l'interface Colab.

import os
from pathlib import Path

if not Path('run_experiments.py').exists():
    print('⚠ run_experiments.py introuvable')
    print('  → Option A : upload via l\'icône 📁 (Files) dans la barre latérale Colab')
    print('  → Option B : monter Google Drive avec:')
    print('      from google.colab import drive')
    print('      drive.mount("/content/drive")')
    print('      !cp /content/drive/MyDrive/CSTL/run_experiments.py .')
else:
    size_kb = Path('run_experiments.py').stat().st_size // 1024
    print(f'✓ run_experiments.py présent ({size_kb} KB)')

In [ ]:
# Cellule 3 — Configuration clés API
# Ne PAS mettre les vraies clés directement dans ce notebook.
# Utilise les "Secrets" de Colab (🔑 icône clé dans la barre latérale).

import os

try:
    from google.colab import userdata
    for key in ['ANTHROPIC_API_KEY', 'OPENAI_API_KEY', 'GOOGLE_API_KEY']:
        try:
            os.environ[key] = userdata.get(key)
            print(f'✓ {key} chargée depuis Colab Secrets')
        except Exception:
            print(f'○ {key} non définie (normal si tu ne testes pas ce provider)')
except ImportError:
    print('Pas dans Colab — assure-toi que les variables d\'environnement sont définies')

---

## Phase 1a — Sanity check avec le client mock

Valide que le harness tourne de bout en bout sans aucun appel API. **Doit afficher fidélité = 1.00 partout** (le mock est déterministe).

In [ ]:
# Cellule 4 — Run sanity (mock, gratuit, <1 min)

!python3 run_experiments.py \
    --models mock \
    --seeds 1,2,3 \
    --n-payloads 20 \
    --run-id sanity_mock

In [ ]:
# Cellule 5 — Vérifier les chiffres du sanity

import pandas as pd

summary = pd.read_csv('results/sanity_mock/summary.csv')
print(summary.to_string(index=False))

# Vérifications attendues
assert (summary['mean'] >= 0.99).all(), 'mock doit donner fidélité ≈ 1.00'
print('\n✓ Sanity check passé — le harness fonctionne correctement')

---

## Phase 1b — Pilote Claude (10 payloads, coût ~$0.10)

**Objectif** : obtenir les premiers *vrais* chiffres CSTL vs JSON sur Claude, avant d'engager le budget complet.

Si cette cellule échoue (erreur 401, quota, etc.), résous le problème ici avant d'aller plus loin.

In [ ]:
# Cellule 6 — Pilote Claude minimal
# 10 payloads × 1 seed × 2 protocoles = 20 appels Claude (~$0.10)

!python3 run_experiments.py \
    --models claude \
    --seeds 1 \
    --n-payloads 10 \
    --experiments fidelity \
    --run-id pilot_claude

In [ ]:
# Cellule 7 — Résultats pilote Claude

import pandas as pd

summary = pd.read_csv('results/pilot_claude/summary.csv')
print('=== Résumé pilote Claude ===')
print(summary.to_string(index=False))

raw = pd.read_csv('results/pilot_claude/raw.csv')
print(f'\nTotal trials : {len(raw)}')
print(f'Erreurs      : {(raw["error"].astype(str).str.len() > 0).sum()}')
print(f'Temps moyen  : {raw["wallclock_s"].mean():.2f} s/trial')

In [ ]:
# Cellule 8 — Inspection qualitative des échecs (si présents)
# Regarder à quoi ressemble un échec permet de diagnostiquer :
#  (a) un vrai problème de fidélité CSTL
#  (b) un problème de prompt système
#  (c) un problème de métrique (line-match trop strict)

import pandas as pd

raw = pd.read_csv('results/pilot_claude/raw.csv')
failures = raw[raw['fidelity'] < 1.0]

if len(failures) == 0:
    print('✓ Aucun échec — fidélité 100% sur le pilote')
else:
    print(f'⚠ {len(failures)} trial(s) avec fidélité < 1.0')
    print(failures[['protocol', 'family', 'fidelity']].to_string(index=False))

---

## Phase 1c — Point de décision GO / NO-GO

**Lis ci-dessous avant de continuer.** Cette décision est le test de réalité le plus important de tout le plan.

In [ ]:
# Cellule 9 — Critère de décision

import pandas as pd

summary = pd.read_csv('results/pilot_claude/summary.csv')
fid = summary[summary['experiment'] == 'fidelity']

cstl_mean = float(fid[fid['protocol'] == 'cstl']['mean'].iloc[0])
json_mean = float(fid[fid['protocol'] == 'json']['mean'].iloc[0])
delta = cstl_mean - json_mean

print(f'CSTL fidelity  : {cstl_mean:.4f}')
print(f'JSON fidelity  : {json_mean:.4f}')
print(f'Delta          : {delta:+.4f}')
print()

if delta >= 0.01:
    print('→ GO : CSTL bat JSON baseline. Lance Phase 1d (production run).')
elif delta >= -0.01:
    print('→ NUANCE : CSTL ≈ JSON sur fidélité. Repositionne le papier sur')
    print('    (a) compression, (b) interprétabilité, (c) unicité k=9.')
    print('  La production run reste utile mais pour d\'autres arguments.')
else:
    print('→ STOP : CSTL perd contre JSON baseline. Avant de dépenser plus,')
    print('  il faut comprendre pourquoi :')
    print('    - Prompt système trop vague ?')
    print('    - Payloads artificiels qui avantagent JSON ?')
    print('    - Claude surajusté sur JSON dans son training ?')
    print('  Inspecter raw.csv + errors.csv avant de continuer.')

---

## Phase 1d — Production run (3 modèles, 5 seeds, 100 payloads)

**Ne lance que si le pilote Claude ci-dessus était concluant.** Cette cellule coûte ~$15-30 et prend 30-60 min.

In [ ]:
# Cellule 10 — Production run complète
# 3 modèles × 5 seeds × 100 payloads × 2 protocoles = 3 000 appels fidélité
# + fictional Korthax + Velundra = ~1 200 appels
# + compression (offline, gratuit)
#
# Décommenter la ligne pour lancer.

# !python3 run_experiments.py \
#     --models claude,gpt4,gemini \
#     --seeds 1,2,3,4,5 \
#     --n-payloads 100 \
#     --temperature 0.0 \
#     --experiments fidelity,fictional,compression \
#     --run-id production_v1

print('Cellule commentée par sécurité. Décommente la commande ci-dessus pour lancer.')

In [ ]:
# Cellule 11 — Chargement des résultats de production
# Si tu n'as pas lancé Phase 1d, on retombe sur les résultats du pilote.

import pandas as pd
from pathlib import Path

RUN_DIR = Path('results/production_v1')
if not RUN_DIR.exists():
    print('⚠ production_v1 introuvable, utilisation de pilot_claude')
    RUN_DIR = Path('results/pilot_claude')

raw = pd.read_csv(RUN_DIR / 'raw.csv')
summary = pd.read_csv(RUN_DIR / 'summary.csv')

print(f'Run utilisée : {RUN_DIR.name}')
print(f'Total trials : {len(raw)}')
print(f'Modèles      : {sorted(raw["model"].unique())}')
print(f'Expériences  : {sorted(raw["experiment"].unique())}')

---

## Phase 2 — Analyse & figures

Génère les tableaux et figures qui iront dans le papier arXiv.

In [ ]:
# Cellule 12 — Table principale du papier : fidélité par modèle × protocole

import pandas as pd

fid = summary[summary['experiment'] == 'fidelity'].copy()
pivot = fid.pivot_table(index='model', columns='protocol',
                        values=['mean', 'std', 'ci95_lo', 'ci95_hi'])
print('=== Table 1 : Fidélité de round-trip ===')
print(pivot.to_string())

# Format prêt pour LaTeX
print('\n=== Version LaTeX-ready ===')
for model in sorted(fid['model'].unique()):
    row = fid[fid['model'] == model]
    cstl = row[row['protocol'] == 'cstl'].iloc[0]
    json_r = row[row['protocol'] == 'json'].iloc[0]
    print(f'{model:10s}  CSTL: {cstl["mean"]:.3f} ± {cstl["std"]:.3f}    '
          f'JSON: {json_r["mean"]:.3f} ± {json_r["std"]:.3f}')

In [ ]:
# Cellule 13 — Figure 1 : fidélité CSTL vs JSON par modèle

import matplotlib.pyplot as plt
import numpy as np

fid = summary[summary['experiment'] == 'fidelity'].copy()
models = sorted(fid['model'].unique())
x = np.arange(len(models))
width = 0.35

cstl_means = [float(fid[(fid['model']==m) & (fid['protocol']=='cstl')]['mean'].iloc[0]) for m in models]
cstl_stds  = [float(fid[(fid['model']==m) & (fid['protocol']=='cstl')]['std'].iloc[0]) for m in models]
json_means = [float(fid[(fid['model']==m) & (fid['protocol']=='json')]['mean'].iloc[0]) for m in models]
json_stds  = [float(fid[(fid['model']==m) & (fid['protocol']=='json')]['std'].iloc[0]) for m in models]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - width/2, cstl_means, width, yerr=cstl_stds, label='CSTL', capsize=4)
ax.bar(x + width/2, json_means, width, yerr=json_stds, label='JSON (baseline)', capsize=4)
ax.set_ylabel('Fidélité (F1 line-level)')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 1.05)
ax.legend()
ax.set_title('Figure 1 — Fidélité de round-trip : CSTL vs JSON baseline')
plt.tight_layout()
plt.savefig(RUN_DIR / 'fig1_fidelity.png', dpi=150)
plt.show()
print(f'✓ Figure sauvée dans {RUN_DIR}/fig1_fidelity.png')

In [ ]:
# Cellule 14 — Figure 2 : fidélité par famille de symboles CSTL
# Révèle quelles familles de la Couche 1 sont fragiles (la ψ, souvent)

import matplotlib.pyplot as plt

cstl_only = raw[(raw['experiment']=='fidelity') & (raw['protocol']=='cstl')]
by_family = cstl_only.groupby('family')['fidelity'].agg(['mean', 'std', 'count']).sort_values('mean')

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(by_family.index, by_family['mean'], xerr=by_family['std'], capsize=4)
ax.set_xlabel('Fidélité (F1)')
ax.set_xlim(0, 1.05)
ax.set_title('Figure 2 — Fidélité CSTL par famille de symboles (Couche 1)')
plt.tight_layout()
plt.savefig(RUN_DIR / 'fig2_by_family.png', dpi=150)
plt.show()
print(by_family)

In [ ]:
# Cellule 15 — Domaines fictifs Korthax & Velundra (anti-leakage)

import pandas as pd

fict = summary[summary['experiment'].str.startswith('fictional')]
if len(fict) > 0:
    print('=== Table 2 : Domaines fictifs ===')
    print(fict[['experiment','model','n','mean','std','ci95_lo','ci95_hi']].to_string(index=False))
else:
    print('○ Pas de données fictional dans cette run — relance avec --experiments fictional')

In [ ]:
# Cellule 16 — Analyse des erreurs pour §5.5 du papier

import pandas as pd

errors = pd.read_csv(RUN_DIR / 'errors.csv')
print(f'Nombre d\'erreurs API : {len(errors)}')

fails = raw[(raw['experiment']=='fidelity') & (raw['fidelity'] < 1.0)]
print(f'\nTrials avec fidélité < 1.0 : {len(fails)}')

if len(fails) > 0:
    print('\nDistribution par famille :')
    print(fails.groupby(['protocol', 'family']).size().to_string())

---

## Phase 3 — Export livrables

Bundle tous les artefacts prêts à déposer sur GitHub / Zenodo.

In [ ]:
# Cellule 17 — Snapshot des résultats + métadonnées pour reproductibilité

import json
from pathlib import Path

meta = json.loads((RUN_DIR / 'run_metadata.json').read_text())
print('=== Métadonnées de run (à citer dans le papier) ===')
print(json.dumps(meta, indent=2))

In [ ]:
# Cellule 18 — Archive zip des résultats pour dépôt Zenodo / supplementary

import shutil
from pathlib import Path

archive = f'cstl_results_{RUN_DIR.name}.zip'
shutil.make_archive(archive.replace('.zip',''), 'zip', RUN_DIR)
print(f'✓ Archive créée : {archive}')
print('\nContenu :')
for p in sorted(Path(RUN_DIR).iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f'  {p.name}  ({size_kb:.1f} KB)')

---

## Prochaines étapes (hors notebook)

1. **Télécharger** `cstl_results_*.zip` (icône 📁 Files dans Colab → clic droit → Download).
2. **Mettre à jour le draft arXiv v2** avec les vrais chiffres (tableau section 5.1, fidélité par famille en 5.5).
3. **Générer les figures manquantes** : pipeline 4-étapes (TikZ), gène k=9 (SVG). Pas dans ce notebook — à faire dans un outil de dessin.
4. **Rédiger le running example Korthax** : prendre 1 payload de `raw.csv` fictional_korthax et l'insérer verbatim dans §4.5.
5. **Postage preprint arXiv** dès que v3 du papier est cohérente avec les chiffres mesurés.

Bonne chance ! 🚀